# 02 — Exploratory Data Analysis

This notebook explores pricing, room types, boroughs, hosts, availability, and review activity.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "airbnb_nyc_cleaned.csv"
IMAGE_DIR = PROJECT_ROOT / "images"
IMAGE_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()


## Descriptive statistics

In [ ]:
price_summary = df["price"].agg(["count", "mean", "median", "std", "min", "max"])
price_summary


## Price distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df.loc[df["price"] <= df["price"].quantile(0.99), "price"], bins=50)
plt.title("Price Distribution (Excluding Top 1% Outliers)")
plt.xlabel("Price (USD)")
plt.ylabel("Listings")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "price_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


## Average price by borough and room type

In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(
    data=df,
    x="neighbourhood_group",
    y="price",
    hue="room_type",
    errorbar=None
)
plt.title("Average Listing Price by Borough and Room Type")
plt.xlabel("Borough")
plt.ylabel("Average Price (USD)")
plt.legend(title="Room Type")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "price_by_borough_room_type.png", dpi=300, bbox_inches="tight")
plt.show()


## Review activity by borough

In [ ]:
reviews = (
    df.groupby("neighbourhood_group", as_index=False)["number_of_reviews"]
    .sum()
    .sort_values("number_of_reviews", ascending=False)
)
reviews


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=reviews, x="neighbourhood_group", y="number_of_reviews")
plt.title("Total Reviews by Borough")
plt.xlabel("Borough")
plt.ylabel("Total Reviews")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "reviews_by_borough.png", dpi=300, bbox_inches="tight")
plt.show()


## Availability by borough

In [ ]:
availability = (
    df.groupby("neighbourhood_group", as_index=False)["availability_365"]
    .mean()
    .sort_values("availability_365", ascending=False)
)
availability


## Top hosts by listing count

In [ ]:
top_hosts = (
    df.groupby(["host_id", "host_name"], as_index=False)
    .size()
    .rename(columns={"size": "listing_count"})
    .sort_values("listing_count", ascending=False)
    .head(10)
)
top_hosts


## Geographic distribution

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(df["longitude"], df["latitude"], s=3, alpha=0.35)
plt.title("Geographic Distribution of Airbnb Listings")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "listing_map.png", dpi=300, bbox_inches="tight")
plt.show()
